# Bygge bilderedigeringsapplikasjoner (OpenAI)

Det er mer med LLM-er enn tekstgenerering. Du kan også generere bilder fra tekstbeskrivelser. Bilder som modalitet er nyttig innen MedTech, arkitektur, turisme, spillutvikling, markedsføring og mer. I denne leksjonen jobber vi med dagens **GPT Image**-modeller på OpenAI-plattformen.

## Læringsmål

Innen slutten av denne leksjonen vil du kunne:

- Forklare hva bildegenerering er og hvor det er nyttig.
- Forstå `gpt-image` modellfamilien og hvordan den skiller seg fra de eldre DALL·E-modellene.
- Bygge en bildegenereringsapplikasjon og redigere bilder.

## Hva er bildegenerering?

Bildegenereringsmodeller lager bilder fra en tekstprompt. Moderne modeller som `gpt-image` lærer forholdet mellom tekst og bilder under trening, og forvandler deretter iterativt tilfeldig støy til et bilde som samsvarer med prompten din.

To velkjente familier av bildemodeller er:

- **`gpt-image` (OpenAI)** — nåværende generasjon brukt i denne leksjonen. Den støtter tekst-til-bilde generering og bilde-redigering (inpainting med en maske).
- **Midjourney** — en populær tredjepartsmodell med egen tjeneste og Discord-basert arbeidsflyt.

> De eldre OpenAI bilde-modellene — **DALL·E 2** og **DALL·E 3** — er eldre. DALL·E 3 er ikke lenger tilgjengelig for nye distribusjoner, og funksjoner som `create_variation` fantes bare i DALL·E 2. Bruk `gpt-image` modellene for nye applikasjoner.

> **Viktig:** `gpt-image` modellene returnerer det genererte bildet som **base64** (`b64_json`), ikke som en URL. Koden din dekoder base64-strengen til bytes og lagrer den — det finnes ingen bilde-URL å laste ned.


## Bygge din første bildegenereringsapplikasjon

Så hva kreves for å bygge en bildegenereringsapplikasjon? Du trenger følgende biblioteker:

- **python-dotenv**, det anbefales sterkt å bruke dette biblioteket for å holde dine hemmeligheter i en *.env*-fil adskilt fra koden.
- **openai**, dette biblioteket vil du bruke for å samhandle med OpenAI API.
- **pillow**, for å arbeide med bilder i Python.


1. Lag en fil *.env* med følgende innhold:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Samle bibliotekene ovenfor i en fil kalt *requirements.txt* slik:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Deretter lager du et virtuelt miljø og installerer bibliotekene:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> For Windows, bruk følgende kommandoer for å opprette og aktivere ditt virtuelle miljø:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Legg til følgende kode i en fil kalt *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Opprett OpenAI-objekt (leser OPENAI_API_KEY fra din .env)
    client = openai.OpenAI()


    try:
        # Lag et bilde ved å bruke bildegenererings-API-en
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Skriv inn din prompttekst her
            size='1024x1024',
            n=1
        )
        # Sett katalogen for det lagrede bildet
        image_dir = os.path.join(os.curdir, 'images')

        # Hvis katalogen ikke eksisterer, opprett den
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialiser bildefilstien (merk at filtypen skal være png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-bilde modeller returnerer bildet som base64 (b64_json), ikke en URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Vis bildet i standard bildeviser
        image = Image.open(image_path)
        image.show()

    # fang unntak
    except openai.BadRequestError as err:
        print(err)

    ```

La oss forklare denne koden:

- Først importerer vi bibliotekene vi trenger, inkludert OpenAI-biblioteket, dotenv-biblioteket, base64-modulen og Pillow-biblioteket.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Etter det oppretter vi klienten, som leser API-nøkkelen fra din ``.env``.

    ```python
    # Opprett OpenAI-objekt
    client = openai.OpenAI()
    ```

- Deretter genererer vi bildet:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Skriv inn tekst for forespørselen din her
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` modeller returnerer bildet som en **base64**-streng i `data[0].b64_json`. Vi dekoder den til bytes og skriver det til en fil — det finnes ingen URL å laste ned fra.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Til slutt åpner vi bildet og bruker standard bildeviser for å vise det:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Flere detaljer om bildegenerering

La oss se på parameterne til `images.generate`:

- **model**, er bildemodellen som skal brukes (for eksempel `gpt-image-1`).
- **prompt**, er tekstprompten som brukes for å generere bildet. Her er det "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, er størrelsen på det genererte bildet (for eksempel `1024x1024`, `1536x1024`, `1024x1536`, eller `"auto"`).
- **n**, er antall genererte bilder. Her genererer vi ett.

> Bildemodeller tar ikke en `temperature`-parameter — det er en kontroll for tekstgenerering. For å få variasjon, kall APIen på nytt; for å redusere variasjon, gjør prompten din mer spesifikk.

## Ytterligere muligheter i bildegenerering

Du har sett hvordan man genererer et bilde med noen få linjer Python. `gpt-image`-modeller kan også **redigere** et eksisterende bilde. Ved å gi et bilde, en valgfri **maske** (som markerer området som skal endres), og en prompt, kan du endre en del av et bilde — for eksempel legge til en hatt på kaninen vår.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# endringer returneres også som base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Grunnbildet inneholder bare kaninen; sluttbildet har hatten på kaninen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
